In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Coffee futures

**New concept: `pct_change()`**

`pct_change()` computes the percentage change from the previous row:

```python
s.pct_change()   # (current - previous) / previous
                 # first row is always NaN — no previous value
```

It's the standard way to turn a price series into a returns series before applying smoothers. The result is in decimal form: `0.02` means +2%.

---

Daily coffee futures prices over 25 trading days. Use `pct_change()` to get daily returns, then use `ewm` to track trend and volatility.

1. Add `returns = price.pct_change()`. What's the mean and std of daily returns? Use `np.nanmean` and `np.nanstd`.
2. Add `ewm_ret = returns.ewm(span=5).mean()`. Count how many days had positive momentum (`ewm_ret > 0`).
3. Add `ewm_vol = returns.ewm(span=5).std()`. Which day had peak volatility? Use `np.nanargmax` — it skips NaN unlike `np.argmax`.

In [7]:
coffee = pd.DataFrame({
    'date':  pd.date_range('2023-03-01', periods=25, freq='B'),
    'price': [184.2, 186.5, 183.1, 188.7, 191.4, 189.2, 194.6, 192.3, 197.8, 201.2,
              198.5, 195.1, 200.4, 204.8, 202.1, 207.6, 211.3, 208.9, 214.2, 218.7,
              215.4, 220.1, 217.8, 223.5, 228.2],
})

# Your code here

coffee['returns'] = coffee['price'].pct_change()
print('mean is: ', np.nanmean(coffee['returns']))
print('std is: ', np.nanstd(coffee['returns']))

coffee['ewm_ret'] = coffee['returns'].ewm(span=5).mean()
print((coffee['ewm_ret']>0).sum(),'had positive momentum')

coffee['ewm_vol'] = coffee['returns'].ewm(span=5).std()
print(coffee.iloc[np.nanargmax(coffee['ewm_vol'])]['date'],'had peak volatility')


mean is:  0.009128990454313743
std is:  0.01815204866446579
22 had positive momentum
2023-03-06 00:00:00 had peak volatility


---

## Level 2 — Online store revenue

18 months of monthly revenue for an online store. No sub-questions this time — write your own analysis.

**Your task:** characterise the store's growth trajectory. At minimum:
- Compute month-over-month growth rate with `pct_change()`
- Identify the best and worst months for growth
- Use `ewm` to describe recent momentum
- Use `np.percentile` somewhere in your analysis

What's the headline finding?

In [15]:
store = pd.DataFrame({
    'month':   pd.date_range('2022-01', periods=18, freq='MS'),
    'revenue': [124000, 118500, 131200, 143800, 138600, 152400,
                147100, 161800, 158300, 174200, 169500, 186300,
                181200, 198700, 193400, 212600, 207800, 228500],
})

# Your code here
store['mom'] = store['revenue'].pct_change()
print('best month is: ', store.loc[store['mom'].idxmax(),'month'])
print('worst month is: ', store.loc[store['mom'].idxmin(),'month'])

store['ewm_re'] = store['revenue'].ewm(span=5).mean()

q = np.percentile(store['revenue'], q = [25,75])

print(store.loc[store['revenue']>q[1],'month'], 'are above the Q3')

best month is:  2022-03-01 00:00:00
worst month is:  2022-02-01 00:00:00
13   2023-02-01
14   2023-03-01
15   2023-04-01
16   2023-05-01
17   2023-06-01
Name: month, dtype: datetime64[ns] are above the Q3


---

## Level 3 — Three delivery zones

Weekly order counts for three delivery zones over 10 weeks. Each zone has a different growth pattern.

1. Add a `wow_growth` column (week-over-week % change) using `groupby('zone').transform(lambda x: x.pct_change())`. Which zone × week had the single biggest jump?
2. Add an `ewm_orders` column using `groupby('zone').transform(lambda x: x.ewm(span=3).mean())`. Which zone is trending highest at week 10?
3. Extract each zone's `count` as a plain array and use `np.corrcoef` to build a 3×3 correlation matrix. Which two zones are most correlated?
4. Use `np.argsort` on the three zones' final `ewm_orders` values to rank them. Use `expanding().mean()` on the full `count` column to add a cumulative average — what does it look like for each zone?

In [67]:
orders = pd.DataFrame({
    'week':  list(pd.date_range('2023-01-02', periods=10, freq='W')) * 3,
    'zone':  ['North']*10 + ['South']*10 + ['East']*10,
    'count': [
        420, 445, 438, 462, 478, 455, 491, 508, 496, 524,   # North: steady growth
        680, 712, 648, 734, 695, 718, 672, 741, 708, 756,   # South: volatile but high
        310, 318, 305, 322, 315, 308, 319, 328, 356, 389,   # East: flat then late surge
    ],
})

# Your code here


orders['wow_growth'] = orders.groupby('zone')['count'].transform(lambda x: x.pct_change())

print(orders.loc[orders['wow_growth'].idxmax(),['week','zone']],'had the single biggest jump')

orders['ewm_orders'] = orders.groupby('zone')['count'].transform(lambda x: x.ewm(span = 3).mean())

ii = orders.loc[orders['week'] == orders['week'].unique()[9]]
print(ii.loc[ii['ewm_orders'].idxmax(),'zone'],'is trending the highest')
zones = orders['zone'].unique()

ca0 = orders.loc[orders['zone'] == zones[0],'count'].to_numpy()
ca1 = orders.loc[orders['zone'] == zones[1],'count'].to_numpy()
ca2 = orders.loc[orders['zone'] == zones[2],'count'].to_numpy()
cc = np.corrcoef([ca0, ca1, ca2])
print(cc)

for i in [0,1,2]:
    for j in [0,1,2]:
        if i>=j:
            cc[i,j] = -np.inf
ccm = pd.DataFrame(cc)
ccm = ccm.unstack()
print(zones[ccm.idxmax()[0]],zones[ccm.idxmax()[1]],'has the highest correlation')


orders['cum_mean'] = orders.groupby('zone')['count'].transform(lambda x: x.expanding().mean())
ss = np.argsort(orders.loc[orders['week'] ==orders['week'].unique()[9], 'ewm_orders'])

[zones[i] for i in ss]

week    2023-01-29 00:00:00
zone                  South
Name: 13, dtype: object had the single biggest jump
South is trending the highest
[[1.         0.58874415 0.76544779]
 [0.58874415 1.         0.62448355]
 [0.76544779 0.62448355 1.        ]]
East North has the highest correlation


['East', 'North', 'South']

In [65]:
orders['week'].unique()

<DatetimeArray>
['2023-01-08 00:00:00', '2023-01-15 00:00:00', '2023-01-22 00:00:00',
 '2023-01-29 00:00:00', '2023-02-05 00:00:00', '2023-02-12 00:00:00',
 '2023-02-19 00:00:00', '2023-02-26 00:00:00', '2023-03-05 00:00:00',
 '2023-03-12 00:00:00']
Length: 10, dtype: datetime64[ns]